# Task 3 — GPU run and resume

Complete the CPU notebook first. Choose a **GPU** runtime and run both cells. Upload the same package ZIP. The fixed queue runs the common probe, standard 20-update continuation, two independent 8-update forks and all four 200-prompt evaluations. Every completed unit persists to Drive. A 120-minute scheduling boundary pauses between units; run this notebook again to resume. A current unit may overrun the boundary. Errors retain the runtime; successful saves and Drive flushing permit the approved automatic disconnect.

No reported result exists until its job actually runs. The standard endpoint is reserved for Task 4. New experimental choices require explicit approval.

In [ ]:
import time
SESSION_START_EPOCH = time.time()
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path
from google.colab import files, drive
uploaded = files.upload()  # Upload ATML_PA2_Task3_Colab.zip.
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly the Task 3 package ZIP.'
assert Path(archives[0]).name == archives[0], 'Upload a filename without directories.'
archive_path = Path('/content') / archives[0]
archive_path.write_bytes(uploaded[archives[0]])
del uploaded
package = Path('/content/task3_package')
package.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as z:
    names = z.namelist()
    assert len(names) == len(set(names)), 'Duplicate archive members.'
    manifest = json.loads(z.read('PACKAGE_SHA256.json'))
    assert set(names) == set(manifest) | {'PACKAGE_SHA256.json'}, 'Package file set differs.'
    for item in z.infolist():
        target = (package / item.filename).resolve()
        assert package.resolve() in target.parents, 'Unsafe archive path.'
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Archive contains a link.'
    for name,digest in manifest.items():
        payload = z.read(name)
        assert hashlib.sha256(payload).hexdigest() == digest, 'Package checksum differs: ' + name
        target = package / name
        target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists():
            assert target.read_bytes() == payload, 'Existing package differs; start a fresh Colab runtime.'
        else:
            target.write_bytes(payload)
expected_source = 'f87021b8ee1057d2afbae29d12c5bd0e904d10b1e9888d9476f99f2959a05c07'
assert hashlib.sha256((package/'task3/source/docs/SOURCE_SHA256.json').read_bytes()).hexdigest() == expected_source
runtime = Path('/content/task3_runtime/course')
subprocess.run([sys.executable,str(package/'task3/restore_source.py'),'--destination',str(runtime)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(runtime/'requirements.txt')],check=True)
drive.mount('/content/drive')
EXPERIMENT = '/content/drive/MyDrive/ATML-Assignment-2/task3/experiment_v1'
os.chdir(runtime)
print('Sealed Task 3 source ready. Scientific settings are fixed in the approved package.')

In [ ]:
# Resume is contract-checked; this never starts a replacement experiment.
command = [sys.executable,'-u','-m','task3_grpo.pipeline','--experiment',EXPERIMENT,
           '--local','/content/task3_runtime/results','--resume',
           '--session-start-epoch',str(SESSION_START_EPOCH)]
process = subprocess.run(command)  # Returns only after the model process exits.
if process.returncode not in (0,75):
    raise RuntimeError('Task 3 stopped with an error. Completed units remain on Drive. Runtime retained for inspection.')
safe = json.loads((Path(EXPERIMENT)/'SAFE_TO_DISCONNECT.json').read_text())
assert safe['created_epoch'] >= SESSION_START_EPOCH
assert safe['model_jobs_returned'] and safe['all_committed_archives_verified']
assert safe['source']['source_manifest_sha256'] == expected_source
for name,digest in safe['archives_sha256'].items():
    h = hashlib.sha256()
    with (Path(EXPERIMENT)/name).open('rb') as f:
        for block in iter(lambda:f.read(2**20),b''):
            h.update(block)
    assert h.hexdigest() == digest, 'Drive changed after process exit: ' + name
print(safe['status'])
if safe['status'] == 'COMPLETE':
    print('Results: '+EXPERIMENT+'/task3_small_results.zip')
else:
    print('120-minute scheduling boundary reached. Run this notebook again to resume saved work.')
# Approved automatic disconnect: verification, stopped process, then successful flush.
# If any earlier assertion or flush fails, unassign is never reached.
drive.flush_and_unmount()
from google.colab import runtime as colab_runtime
colab_runtime.unassign()